# 失败重试

**常见用法**：瞬态故障自动重试（网络抖动、限流、下游超时）；
生产上常配指数退避（`time.sleep` 间隔递增）和异常类型过滤（只重试 `ConnectionError`/`Timeout`）。

**钩子内的做法**：
- 循环调 `execute(request)`——官方注释明确 execute 可多次调用，每次相互独立
- 成功必须**立即 `return`**（漏 return 会把成功结果丢弃，实测踩过）
- 次数用尽 → 回填 `status="error"` 的 ToolMessage 把错误告诉模型，不抛出炸图

In [13]:
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt.tool_node import ToolCallRequest  # 注意：不在 prebuilt 顶层导出
from langgraph.types import Command, interrupt
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

FAILS = {"count": 0}


# 定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    raise ConnectionError("连接超时")
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    FAILS["count"] += 1
    if FAILS["count"] <= 2:
        raise ConnectionError("网络抖动")
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


class ChatState(MessagesState):
    pass


def wrap_tool_call(request: ToolCallRequest, execute, retries: int = 5) -> ToolMessage | object:
    """失败自动重试，重试次数用尽后把错误回填给模型"""
    tc = request.tool_call  # 单个 dict：{'name','args','id'}
    for i in range(retries):
        print(f"[{tc['name']}({tc['args']})")  # try 外：打印自身的错不算工具失败
        try:
            return execute(request)
        except Exception as e:
            print(f"{tc['name']} 调用失败")
            error = e
    return ToolMessage(
        content=f"重试 {retries} 次仍失败: {error}",
        name=tc["name"],
        tool_call_id=tc["id"],
        status="error"
    )


# 工具节点
tool_node = ToolNode(tools, wrap_tool_call=wrap_tool_call)


# 定义状态
class State(MessagesState):
    pass


# 模型节点
def llm_node(state: ChatState) -> State:
    ai_msg = model_with_tools.invoke(state["messages"])
    return {"messages": [ai_msg]}


def router(state: ChatState) -> Literal["tool_node", END]:
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])
builder.add_edge("tool_node", "llm_node")

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}
graph = builder.compile(checkpointer)
res = graph.invoke(
    {
        "messages":
            [
                SystemMessage("你是一个助手, 工具调用失败请重试"),
                HumanMessage("帮我查一下北京的天气,以及体育方面的新闻")
            ]
    },
    config=config
)
print(res)

[get_weather({'city': '北京'})

get_weather 调用失败

[get_news({'topic': '体育'})

[get_weather({'city': '北京'})

get_weather 调用失败

...

[get_weather({'city': '北京'})

get_weather 调用失败

[get_weather({'city': '北京'})

get_weather 调用失败

{
    'messages': [
        SystemMessage(
            content='你是一个助手, 工具调用失败请重试',
            additional_kwargs={},
            response_metadata={},
            id='87b0ab10-5ab0-4349-b827-73aa2ebc3be9'
        ),
        HumanMessage(
            content='帮我查一下北京的天气,以及体育方面的新闻',
            additional_kwargs={},
            response_metadata={},
            id='2e9281d1-49f0-4277-834e-f5fcee709366'
        ),
        AIMessage(
            content='我来同时帮您查询北京天气和体育新闻。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 77,
                    'prompt_tokens': 357,
                    'total_tokens': 434,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 229
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '4382cbef-6719-4453-9dc4-510bedb3c5c8',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b0b7-1239-72c0-bddc-8ac37eaa11b1-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_99FbKfjXsXsdTRxPchvO5381',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_news',
                    'args': {'topic': '体育'},
                    'id': 'call_01_CDnH9vZsDYkfGRoso2NG0665',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 357,
                'output_tokens': 77,
                'total_tokens': 434,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='重试 5 次仍失败: 连接超时',
            name='get_weather',
            id='364d2587-56ba-486c-9d97-be928ad750d5',
            tool_call_id='call_00_99FbKfjXsXsdTRxPchvO5381',
            status='error'
        ),
        ToolMessage(
            content='最新体育新闻：中国队取得了胜利。',
            name='get_news',
            id='67a7bc03-0407-44f1-aaca-aaf524f28c7d',
            tool_call_id='call_01_CDnH9vZsDYkfGRoso2NG0665'
        ),
        AIMessage(
            content='体育新闻查到了，但北京天气查询暂时失败（连接超时）。让我再试一次。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 58,
                    'prompt_tokens': 474,
                    'total_tokens': 532,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 218
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '63bb200d-7307-43ab-87f7-abd3596c7847',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0b0